# A Machine Learning Approach on Empirical Analysis of Stock Price Forecasting Models

**Internship project context:** Codegnan IT Solutions Pvt. Ltd.  
**Project focus:** Web/data collection, EDA, time-series forecasting, model evaluation, and optional deployment preparation.

> **Integrity note:** This notebook is a reproducible project template, not a record of verified internship results. It downloads market data when run and calculates metrics from the held-out test period. Do not report any results as historical internship achievements unless they match work you actually completed.

## Workflow
1. Load daily historical prices
2. Explore and prepare time-ordered data
3. Build next-day closing-price features
4. Train baseline and machine-learning models without shuffling time
5. Evaluate MAE, RMSE, R², MAPE, and directional accuracy
6. Save predictions and the best model


## 1. Install dependencies

Run this cell in Jupyter Notebook or Google Colab. The data download uses `yfinance`; if you already have a CSV file, you can skip downloading and load your own data in Section 2.

In [ ]:
%pip install -q yfinance pandas numpy matplotlib scikit-learn joblib

## 2. Load stock data

Change `TICKER` and the date range to the stock you want to study. This example uses Apple (`AAPL`) only as a demonstration. You can also set `USE_LOCAL_CSV = True` and provide a CSV containing at least `Date` and `Close` columns.

Use a consistent data source and record the ticker, date range, and retrieval date when reporting results.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKER = "AAPL"                 # Example only; change to the stock studied
START_DATE = "2018-01-01"
END_DATE = None                 # None means up to the latest data available
USE_LOCAL_CSV = False
CSV_PATH = "stock_prices.csv"   # Used only when USE_LOCAL_CSV = True

if USE_LOCAL_CSV:
    raw = pd.read_csv(CSV_PATH)
    if "Date" not in raw.columns or "Close" not in raw.columns:
        raise ValueError("CSV must contain 'Date' and 'Close' columns.")
    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
    data = raw.copy()
else:
    import yfinance as yf
    data = yf.download(
        TICKER,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=False
    )
    if data.empty:
        raise RuntimeError("No data downloaded. Check ticker, dates, or internet access.")
    # Recent yfinance versions may return MultiIndex columns.
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data = data.reset_index()
    if "Date" not in data.columns and "Datetime" in data.columns:
        data = data.rename(columns={"Datetime": "Date"})

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
data["Close"] = pd.to_numeric(data["Close"], errors="coerce")
data = data[["Date", "Close"]].dropna().sort_values("Date")
data = data.drop_duplicates(subset="Date").reset_index(drop=True)

print(f"Rows: {len(data):,}")
print(f"First date: {data['Date'].min().date()}")
print(f"Last date:  {data['Date'].max().date()}")
data.head()

## 3. Exploratory data analysis

The chart and descriptive statistics summarize the selected price series. Historical price levels alone do not establish that a model can predict future prices; evaluation must use unseen, later dates.

In [ ]:
display(data.describe())
plt.figure(figsize=(12, 5))
plt.plot(data["Date"], data["Close"])
plt.title(f"{TICKER} adjusted closing price" if not USE_LOCAL_CSV else "Closing price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

data["Daily_Return_Pct"] = data["Close"].pct_change() * 100
display(data["Daily_Return_Pct"].describe().to_frame("Daily return (%)"))

## 4. Feature engineering

The target is the **next trading day's closing price**. Features are computed from information available at the end of the current day, then shifted so that row *t* uses information available by day *t* to predict day *t+1*. The chronological split below prevents future observations from entering training.

In [ ]:
df = data[["Date", "Close"]].copy()
df["Return_1d"] = df["Close"].pct_change()
df["Lag_1"] = df["Close"].shift(1)
df["Lag_2"] = df["Close"].shift(2)
df["Lag_3"] = df["Close"].shift(3)
df["Lag_5"] = df["Close"].shift(5)
df["MA_5"] = df["Close"].rolling(5).mean()
df["MA_10"] = df["Close"].rolling(10).mean()
df["Volatility_5"] = df["Return_1d"].rolling(5).std()
df["Target_Next_Close"] = df["Close"].shift(-1)
df["Target_Date"] = df["Date"].shift(-1)

feature_cols = [
    "Close", "Return_1d", "Lag_1", "Lag_2", "Lag_3", "Lag_5",
    "MA_5", "MA_10", "Volatility_5"
]
model_data = df.dropna(subset=feature_cols + ["Target_Next_Close"]).reset_index(drop=True)

if len(model_data) < 100:
    print("Warning: Fewer than 100 usable observations; metrics may be unstable.")

X = model_data[feature_cols]
y = model_data["Target_Next_Close"]
target_dates = model_data["Target_Date"]

print("Usable rows:", len(model_data))
print("Features:", feature_cols)
model_data[["Date", "Close", "Target_Date", "Target_Next_Close"]].head()

## 5. Chronological train/test split

The first 80% of observations are used for training and the final 20% for testing. The test set represents later dates than the training set. Avoid random shuffling for this time-series task.

In [ ]:
split_idx = int(len(model_data) * 0.80)
if split_idx < 30 or len(model_data) - split_idx < 10:
    raise ValueError("Not enough data for a meaningful chronological train/test split.")

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
test_dates = target_dates.iloc[split_idx:]

print(f"Training rows: {len(X_train)} ({model_data['Date'].iloc[0].date()} to {model_data['Date'].iloc[split_idx-1].date()})")
print(f"Testing rows:  {len(X_test)} (targets {test_dates.iloc[0].date()} to {test_dates.iloc[-1].date()})")

## 6. Train models

Models included:
- **Persistence baseline:** predicts the next close will equal today's close
- **Linear Regression**
- **Random Forest Regressor**
- **Gradient Boosting Regressor**

The persistence baseline is important: a more complex model should be compared against a simple forecast. Hyperparameter tuning is not included in this starter notebook; any tuning should use time-aware validation on the training period only.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

predictions = {}
# Persistence baseline: current day's close is the forecast for next day's close.
predictions["Persistence Baseline"] = X_test["Close"].to_numpy()

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

print("Models trained:", ", ".join(predictions.keys()))

## 7. Calculate performance metrics

- **MAE:** mean absolute error, in the same units as the stock price
- **RMSE:** root mean squared error, in the same units; larger errors are penalized more
- **R²:** explained variance relative to the test-set mean; it can be negative
- **MAPE:** mean absolute percentage error; reported only where actual values are nonzero
- **Directional accuracy:** percentage of cases where predicted up/down movement matches actual up/down movement, relative to the previous day's close

These are test-set results for the selected data period, not proof of future profitability. A stock-price forecasting model is not automatically a trading strategy.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calculate_metrics(actual, predicted, previous_close):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    previous_close = np.asarray(previous_close, dtype=float)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    nonzero = actual != 0
    mape = (np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100
            if np.any(nonzero) else np.nan)

    actual_direction = np.sign(actual - previous_close)
    predicted_direction = np.sign(predicted - previous_close)
    directional_accuracy = np.mean(actual_direction == predicted_direction) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_pct": mape,
        "Directional_Accuracy_pct": directional_accuracy
    }

previous_close_test = X_test["Close"].to_numpy()
rows = []
for name, pred in predictions.items():
    row = {"Model": name}
    row.update(calculate_metrics(y_test.to_numpy(), pred, previous_close_test))
    rows.append(row)

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(results.round(4))

## 8. Visualize test predictions

Compare predictions against actual values only over the held-out test period.

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(test_dates, y_test.to_numpy(), label="Actual", linewidth=2)
for name, pred in predictions.items():
    plt.plot(test_dates, pred, label=name, alpha=0.8)
plt.title("Actual vs predicted next-day closing prices (test period)")
plt.xlabel("Target date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

best_model_name = results.iloc[0]["Model"]
print("Lowest-RMSE model on this test period:", best_model_name)
print("Choose a model based on the evaluation goal and baseline comparison, not one metric alone.")

## 9. Save predictions and metrics

The CSV files contain the measured results for this run. They can be used to support a resume bullet only after you confirm the dataset and method reflect your actual project work.

In [ ]:
prediction_output = pd.DataFrame({
    "Target_Date": pd.to_datetime(test_dates).to_numpy(),
    "Actual_Next_Close": y_test.to_numpy(),
    "Previous_Close": previous_close_test
})
for name, pred in predictions.items():
    safe_name = name.lower().replace(" ", "_")
    prediction_output[f"Pred_{safe_name}"] = pred

prediction_output.to_csv("stock_forecasting_test_predictions.csv", index=False)
results.to_csv("stock_forecasting_metrics.csv", index=False)
print("Saved stock_forecasting_test_predictions.csv")
print("Saved stock_forecasting_metrics.csv")

## 10. Save a trained model (optional)

This saves the lowest-RMSE **machine-learning model** from the models trained above. If the persistence baseline has the lowest RMSE, the notebook instead saves the best trained ML model among the three ML models, and clearly reports that choice. Model persistence is not the same as deployment; production deployment needs additional data validation, monitoring, and an inference interface.

In [ ]:
import joblib

ml_results = results[results["Model"].isin(models.keys())]
if ml_results.empty:
    raise RuntimeError("No trained ML model available to save.")

best_ml_name = ml_results.iloc[0]["Model"]
best_ml_model = models[best_ml_name]
joblib.dump(
    {
        "model": best_ml_model,
        "feature_columns": feature_cols,
        "ticker": TICKER if not USE_LOCAL_CSV else "local_csv",
        "target": "next trading day's closing price"
    },
    "best_stock_forecasting_model.joblib"
)
print(f"Saved model: best_stock_forecasting_model.joblib")
print(f"Best ML model by test RMSE: {best_ml_name}")

## 11. Resume reporting template

After running the notebook, replace placeholders with values from `stock_forecasting_metrics.csv`, and include the ticker and test period if space allows.

**Template:**  
- Built and compared [models actually used] for next-day stock-price forecasting using historical market data and lag/rolling-window features; evaluated on a chronological holdout using MAE **[value]**, RMSE **[value]**, and R² **[value]**.

Do not claim a specific score, model, AWS deployment, or business impact unless you actually completed and can explain that work. For a fair comparison, disclose the stock/ticker, date range, data source, target definition, and test split. Stock price prediction metrics alone do not indicate trading profitability.